# Phase 5 — Qwen3 base-model benchmark

Runs a frozen, test-only benchmark before QLoRA training. Use a Kaggle GPU session with Internet enabled and attach the completed Phase 4 output.

In [ ]:
import platform
import subprocess
import sys
from pathlib import Path

BRANCH = "refactoring/phase-transformer"
REPOSITORY = "https://github.com/SachinGupta2012/Text-Summarizer.git"
PROJECT_ROOT = Path("/kaggle/temp/text-summarizer/project")
OUTPUT_DIRECTORY = Path("/kaggle/working/text-summarizer/phase5/base-benchmark-v1")

print(f"Python: {platform.python_version()}")
print(f"Platform: {platform.platform()}")
print(f"Output: {OUTPUT_DIRECTORY}")

In [ ]:
requirements = [
    "transformers>=4.57,<6",
    "accelerate>=1.10,<2",
    "bitsandbytes>=0.48,<1",
    "bert-score>=0.3.13,<0.4",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *requirements],
    check=True,
)
print("Benchmark dependencies are ready.")

In [ ]:
if not (PROJECT_ROOT / ".git").is_dir():
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "--single-branch",
            REPOSITORY,
            str(PROJECT_ROOT),
        ],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
ml_source = str(PROJECT_ROOT / "ml" / "src")
if ml_source not in sys.path:
    sys.path.insert(0, ml_source)
print(f"Project commit: {commit}")

In [ ]:
phase4_manifests = sorted(Path("/kaggle/input").rglob("phase4-manifest.json"))
working_manifest = Path("/kaggle/working/text-summarizer/phase4/qwen3-sft-v1/phase4-manifest.json")
if working_manifest.is_file():
    phase4_manifests.append(working_manifest)
phase4_manifests = sorted(set(phase4_manifests))
if len(phase4_manifests) != 1:
    raise RuntimeError(f"Expected exactly one Phase 4 input; found {phase4_manifests}")
PHASE4_DIRECTORY = phase4_manifests[0].parent
print(f"Phase 4 input: {PHASE4_DIRECTORY}")

In [ ]:
from summarizer_ml.evaluation import (
    Phase5Config,
    load_benchmark_selection,
    prepare_benchmark_selection,
)

CONFIG = Phase5Config()
selection_manifest = prepare_benchmark_selection(
    PHASE4_DIRECTORY,
    OUTPUT_DIRECTORY,
    CONFIG,
)
selection = load_benchmark_selection(selection_manifest.selection_path)
print(selection_manifest.model_dump_json(indent=2))
print(CONFIG.model_dump_json(indent=2))

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Phase 5 requires a Kaggle GPU accelerator.")
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {properties.name}; memory={properties.total_memory / (1024**3):.1f} GiB")

In [ ]:
from time import perf_counter

from summarizer_ml.evaluation import GeneratedSummary
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization = BitsAndBytesConfig(
    load_in_4bit=CONFIG.load_in_4bit,
    bnb_4bit_quant_type=CONFIG.quantization_type,
    bnb_4bit_use_double_quant=CONFIG.use_double_quantization,
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(CONFIG.model_id)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    CONFIG.model_id,
    device_map="auto",
    quantization_config=quantization,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model.eval()


class QwenBaselineGenerator:
    def __init__(self, active_model, active_tokenizer) -> None:
        self.model = active_model
        self.tokenizer = active_tokenizer
        revision = getattr(self.model.config, "_commit_hash", None)
        self._model_revision = str(revision or "unknown")

    @property
    def model_revision(self) -> str:
        return self._model_revision

    def generate(self, examples):
        texts = [
            self.tokenizer.apply_chat_template(
                [message.model_dump(mode="json") for message in example.prompt],
                tokenize=False,
                add_generation_prompt=True,
            )
            for example in examples
        ]
        encoded = self.tokenizer(
            texts,
            add_special_tokens=False,
            padding=True,
            return_tensors="pt",
        )
        encoded = {key: value.to(self.model.device) for key, value in encoded.items()}
        input_width = encoded["input_ids"].shape[1]
        torch.cuda.synchronize()
        started = perf_counter()
        with torch.inference_mode():
            output = self.model.generate(
                **encoded,
                do_sample=CONFIG.do_sample,
                max_new_tokens=CONFIG.max_new_tokens,
                use_cache=True,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        torch.cuda.synchronize()
        latency = (perf_counter() - started) / len(examples)
        results = []
        for row in output[:, input_width:]:
            token_ids = row.tolist()
            if self.tokenizer.eos_token_id in token_ids:
                end = token_ids.index(self.tokenizer.eos_token_id) + 1
                token_ids = token_ids[:end]
            results.append(
                GeneratedSummary(
                    text=self.tokenizer.decode(token_ids, skip_special_tokens=True).strip(),
                    generated_tokens=len(token_ids),
                    latency_seconds=latency,
                )
            )
        return tuple(results)


generator = QwenBaselineGenerator(model, tokenizer)
print(f"Model revision: {generator.model_revision}")
print(f"Model device: {model.device}")

In [ ]:
smoke = generator.generate(selection[:2])
for example, generated in zip(selection[:2], smoke, strict=True):
    if not generated.text:
        raise RuntimeError(f"Empty smoke prediction for {example.record_id}")
    print(example.dataset, generated.generated_tokens, generated.text[:300])
print("Base-model generation smoke gate passed.")

In [ ]:
from time import monotonic

from summarizer_ml.evaluation import (
    generate_baseline_predictions,
    regenerate_capped_predictions,
)

generation_started = monotonic()


def report_generation(stage: str, count: int, total: int) -> None:
    elapsed_minutes = (monotonic() - generation_started) / 60
    rate = count / max(elapsed_minutes, 1e-6)
    remaining = (total - count) / max(rate, 1e-6)
    print(
        f"Phase 5 {stage}: {count:,}/{total:,}; "
        f"elapsed={elapsed_minutes:.1f}m; eta={remaining:.1f}m",
        flush=True,
    )


predictions = generate_baseline_predictions(
    selection,
    OUTPUT_DIRECTORY / "base-predictions.jsonl",
    generator,
    CONFIG,
    progress_every=25,
    progress_callback=report_generation,
)
predictions, repaired_record_ids = regenerate_capped_predictions(
    selection,
    OUTPUT_DIRECTORY / "base-predictions.jsonl",
    generator,
    CONFIG,
    previous_max_new_tokens=768,
    progress_every=5,
    progress_callback=report_generation,
)
model_revision = generator.model_revision
print(f"Predictions complete: {len(predictions):,}")
print(f"Capped predictions repaired: {len(repaired_record_ids):,}")

In [ ]:
import gc

del generator
del model
gc.collect()
torch.cuda.empty_cache()
print("Qwen model unloaded before semantic scoring.")

In [ ]:
import json

from bert_score import score as bert_score

bertscore_path = OUTPUT_DIRECTORY / "bertscore.jsonl"
bertscore_by_id = {}
if bertscore_path.is_file():
    bertscore_by_id = {
        row["record_id"]: float(row["f1"])
        for row in (
            json.loads(line)
            for line in bertscore_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        )
    }
selection_by_id = {example.record_id: example for example in selection}
predictions_by_id = {prediction.record_id: prediction for prediction in predictions}
score_record_ids = [
    example.record_id
    for example in selection
    if example.record_id in set(repaired_record_ids)
    or example.record_id not in bertscore_by_id
]
if score_record_ids:
    references = [selection_by_id[record_id].reference_summary for record_id in score_record_ids]
    candidates = [predictions_by_id[record_id].prediction for record_id in score_record_ids]
    _, _, f1 = bert_score(
        candidates,
        references,
        model_type=CONFIG.bertscore_model_id,
        lang="en",
        batch_size=16,
        device="cuda:0",
        rescale_with_baseline=True,
        verbose=True,
    )
    bertscore_by_id.update(
        dict(zip(score_record_ids, map(float, f1.tolist()), strict=True))
    )
    temporary = bertscore_path.with_name(f"{bertscore_path.name}.tmp")
    with temporary.open("w", encoding="utf-8", newline="\n") as output:
        for example in selection:
            record_id = example.record_id
            value = bertscore_by_id[record_id]
            output.write(json.dumps({"record_id": record_id, "f1": value}))
            output.write("\n")
    temporary.replace(bertscore_path)
print(f"BERTScore records updated: {len(score_record_ids):,}")
print(f"BERTScore records: {len(bertscore_by_id):,}")

In [ ]:
from summarizer_ml.evaluation import evaluate_baseline

result = evaluate_baseline(
    selection,
    predictions,
    OUTPUT_DIRECTORY,
    CONFIG,
    selection_manifest.phase4_manifest_sha256,
    model_revision,
    bertscore=bertscore_by_id,
)
print(result.report.model_dump_json(indent=2))
print(f"Phase 5 artifacts: {result.output_directory}")
print("Phase 5 base-model benchmark gate passed.")

In [ ]:
required_artifacts = [
    "selection-manifest.json",
    "benchmark-selection.jsonl",
    "base-predictions.jsonl",
    "bertscore.jsonl",
    "baseline-report.json",
]
for name in required_artifacts:
    path = OUTPUT_DIRECTORY / name
    print(name, "exists:", path.is_file(), "bytes:", path.stat().st_size if path.exists() else 0)
    if not path.is_file():
        raise RuntimeError(f"Missing Phase 5 artifact: {path}")
print("Phase 5 artifact gate passed.")